# Calcula tu inflación — por marca

Calcula la **inflación real por marca** con los datos abiertos de *Quién es Quién
en los Precios* (Profeco), y detecta **reduflación**: cuando el producto cuesta
casi lo mismo pero trae menos contenido.

**No necesitas saber Python ni instalar nada.** Ejecuta las celdas en orden con
el botón ▶ de la izquierda (o `Shift + Enter`).

### Antes de empezar: sube tus datos a Google Drive

Sube a tu Google Drive los archivos de QQP, **sin descomprimir**. Ponlos en una
carpeta llamada `QQP`:

```
Mi unidad/
└── QQP/
    ├── QQP_2024.rar
    └── QQP_2025.rar
```

Subir los `.rar` es mucho más rápido que subir los CSV: el `.rar` pesa unos
100 MB y adentro trae el año completo. Este cuaderno los descomprime por ti.

> Si prefieres, también puedes subir los CSV ya descomprimidos a esa carpeta.
> El cuaderno acepta las dos formas.

---

## Paso 1 — Preparar

Tarda medio minuto.

In [ ]:
#@title Paso 1: preparar (ejecuta esta celda)
!pip install -q pandas
!apt-get -qq install -y unar > /dev/null 2>&1
!wget -q -O inflacion_por_marca.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/inflacion_por_marca.py
!wget -q -O colab_qqp.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/colab_qqp.py

# Se recarga por si ya estaba importado de una ejecucion anterior,
# asi las correcciones llegan sin tener que reabrir el cuaderno.
import importlib, colab_qqp
importlib.reload(colab_qqp)
colab_qqp.preparar()

---

## Paso 2 — Conectar tu Google Drive

Al ejecutar, Google te va a pedir permiso: elige tu cuenta y acepta. Es el acceso
normal de Colab a tu Drive; nada sale de tu sesión.

In [ ]:
#@title Paso 2: conectar Drive (ejecuta y acepta el permiso)
from google.colab import drive
drive.mount("/content/drive")
print("\nDrive conectado.")

---

## Paso 3 — Decir en qué carpeta de Drive están

Si los pusiste en `Mi unidad/QQP`, déjalo como está. Si usaste otro nombre,
cámbialo (respeta mayúsculas y acentos).

In [ ]:
#@title Paso 3: ubicar la carpeta
CARPETA_EN_DRIVE = "QQP"  #@param {type:"string"}

CARPETA = colab_qqp.ubicar(CARPETA_EN_DRIVE)

---

## Paso 4 — Extraer los comprimidos

Escribe los nombres de los `.rar` que quieres usar, **separados por coma**.
Cada uno trae un año completo.

Si ya subiste los CSV sueltos, deja esto vacío `""` y ejecuta: los copia y ya.

> Extraer un año tarda unos minutos. Los archivos se guardan en la sesión de
> Colab, no en tu Drive, así que no te consume espacio.

In [ ]:
#@title Paso 4: extraer
COMPRIMIDOS = "QQP_2024.rar, QQP_2025.rar"  #@param {type:"string"}

colab_qqp.extraer(CARPETA, COMPRIMIDOS)

---

## Paso 5 — Elegir los dos años

QQP **no nombra igual los archivos de todos los años**. Unos traen el mes en el
nombre (`01-2024_01.csv`), otros solo un número de pieza (`012015.csv`, donde el
`01` no es enero sino la pieza 1).

Por eso aquí solo se elige el **año**, con el patrón `*AÑO*.csv`. El mes se filtra
en el Paso 6, leyendo la fecha que traen los datos por dentro — eso sí es
confiable.

| Para el año | Escribes |
|---|---|
| 2024 | `*2024*.csv` |
| 2025 | `*2025*.csv` |

El Paso 4 te dijo qué años tienes disponibles. Copia dos de ahí.

In [ ]:
#@title Paso 5: los dos años
PATRON_BASE   = "*2024*.csv"  #@param {type:"string"}
PATRON_ACTUAL = "*2025*.csv"  #@param {type:"string"}

colab_qqp.revisar(PATRON_BASE, PATRON_ACTUAL)

---

## Paso 6 — Elegir el mes y qué analizar

**El MES es obligatorio.** Si lo dejas en 0, se mezclan todos los meses del año y
el resultado no sirve de nada. Usa el mismo mes en ambos años (se aplica a los
dos), para no confundir inflación con temporada.

`8` = agosto, `1` = enero, y así.

**También pon al menos un filtro de producto.** Un año de QQP trae millones de
precios; sin filtrar se acaba la memoria.

Varios términos van separados por espacios: `"DESODORANTE SHAMPOO"` trae los dos.

In [ ]:
#@title Paso 6: mes y filtros
MES = 8  #@param {type:"slider", min:0, max:12, step:1}

PRODUCTO  = "DESODORANTE"  #@param {type:"string"}
MARCA     = ""             #@param {type:"string"}
CATEGORIA = ""             #@param {type:"string"}
ESTADO    = ""             #@param {type:"string"}
CADENA    = ""             #@param {type:"string"}

POR_CADENA = True      #@param {type:"boolean"}
MIN_OBSERVACIONES = 3  #@param {type:"integer"}

if not MES:
    print("FALTA EL MES: en 0 se mezclan todos los meses y el resultado no sirve.")
elif not any((PRODUCTO, MARCA, CATEGORIA, ESTADO, CADENA)):
    print("FALTA UN FILTRO: sin producto ni categoria se puede acabar la memoria.")
else:
    meses = ["", "enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
             "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
    print(f"Listo: se comparara {meses[MES]} de un anio contra {meses[MES]} del otro.")
    print("Pasa al Paso 7.")

---

## Paso 7 — Calcular

Verás un contador de filas mientras avanza. Con archivos grandes tarda varios
minutos.

In [ ]:
#@title Paso 7: calcular
colab_qqp.calcular(PATRON_BASE, PATRON_ACTUAL,
                   producto=PRODUCTO, marca=MARCA, categoria=CATEGORIA,
                   estado=ESTADO, cadena=CADENA, mes=MES,
                   por_cadena=POR_CADENA, min_obs=MIN_OBSERVACIONES)

---

## Paso 8 — Guardar el resultado

Lo guarda en tu Drive (para que no se pierda) y te lo descarga.

In [ ]:
#@title Paso 8: guardar y descargar
import os, shutil
if "CARPETA" not in dir():
    raise SystemExit("Falta ejecutar el Paso 3 antes que este.")

if not os.path.exists("resultado.csv"):
    print("Todavia no hay resultado. Ejecuta el Paso 7 primero.")
else:
    destino = os.path.join(CARPETA, "resultado.csv")
    shutil.copy("resultado.csv", destino)
    print(f"Guardado en tu Drive: {destino}")
    from google.colab import files
    files.download("resultado.csv")

---

## Cómo leer la tabla de reduflación

| Columna | Qué significa |
|---|---|
| **ETIQUETA** | Cuánto subió el precio que ves en el anaquel |
| **CONTENIDO** | Cuánto cambió el tamaño del empaque (negativo = encogió) |
| **REAL** | Cuánto subió el precio por gramo o mililitro |
| **BRECHA** | REAL menos ETIQUETA — la inflación que no se ve |

Una marca con `<-- ENCOGIO` redujo el contenido. Si además su BRECHA es grande,
estás pagando bastante más por gramo aunque el precio del anaquel casi no se
haya movido.

## Notas

- Se usa la **mediana** de precios, no el promedio, para que unos pocos registros
  mal capturados no distorsionen el resultado.
- Un artículo solo aparece si está en **ambos** periodos con al menos
  `MIN_OBSERVACIONES` registros.
- Los archivos extraídos viven solo mientras dure la sesión de Colab. Tu Drive
  conserva los `.rar` originales y el `resultado.csv`.

Código y documentación: <https://github.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N>